# 08 — Descoberta não concede execução

Referência: [AgentCore workshops](https://github.com/awslabs/agentcore-samples/tree/main/06-workshops). O código e o IaC desta jornada adaptam os conceitos às contas do workshop.

## Pré-requisito: 1 conta AWS

Esta aula não exige fronteiras entre contas. Seus stacks, integrações e permissões precisam estar implantados na conta da plataforma. Ela não funciona automaticamente em uma conta AWS vazia.

**Runner CodeBuild do projeto:** a implantação atual exige quatro contas e as integrações Registry, Terraform e RAG, inclusive com `--labs`. A seleção parcial não converte esse runner em uma implantação de conta única. Confira a [matriz de contas e execução](../docs/contas-e-elegibilidade.md).

![Arquitetura AWS da aula 08](../docs/diagramas/svg/08_registry_descoberta_execucao.svg)

Leia as fronteiras do diagrama e confronte-as com o IaC e as chamadas abaixo.

Todos os serviços e agentes desta jornada são implantados na AWS.
O notebook é o roteiro e cliente das APIs; ferramentas, inferência,
políticas, fontes e memória executam nos serviços indicados.
Use o ambiente AWS descrito no [guia de implantação](../docs/deploy.md).
Configure os identificadores dos stacks antes de executar. Uma
dependência ausente bloqueia a aula antes das chamadas do exercício.
Antes de usar “Run All”, selecione as aulas compatíveis com as contas
preparadas. O bloco 10–14 exige quatro contas na implantação atual.

### Carregar a configuração e conferir as dependências

A configuração contém perfis/roles e IDs, nunca senhas ou tokens. O STS confirma a conta antes de consultar os stacks; o manifesto exige as dependências desta aula.

In [ ]:
import json
import os
import uuid
from pathlib import Path
from IPython.display import Code, display
import inspect
from santander_demo.aws_workshop.client import AWSWorkshop
from santander_demo.aws_workshop.config import load
from santander_demo.aws_workshop.manifest import require

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CONFIG = Path(os.environ.get("WORKSHOP_CONFIG", ROOT / "config/aws.workshop.json"))
config = load(CONFIG)
dependencies = require(config, "08")
print({"lab": "08", "dependencies": list(dependencies), "region": config["region"]})

### Carregar o catálogo implantado

O catálogo tem roles diferentes para desenvolver, publicar, aprovar e consumir. Os SDKs atuais são agent-registry-control para governança e agent-registry para descoberta.

In [ ]:
from santander_demo.registry_workflow.client import RegistryLab
registry_config = json.loads((ROOT / dependencies["registry"]["config"]).read_text())
registry = RegistryLab(registry_config)
assert registry_config["RegistryArn"].startswith("arn:aws:agent-registry:")
print({"registry": registry_config["RegistryArn"], "states": registry.states()})

### Descobrir com uma sessão que não pode invocar

A session policy de STS reduz a sessão do consumidor somente à descoberta. Encontrar um registro não deve conceder InvokeAgentRuntime.

In [ ]:
from botocore.exceptions import ClientError
restricted = registry.persona("Consumer", discovery_only=True)
records = registry.discover(restricted)
record = next(x for x in records if x["recordId"] == registry_config["AgentRecordId"])
try:
    registry.invoke(record, restricted)
except ClientError as error:
    assert error.response["Error"]["Code"] in {"AccessDeniedException", "AccessDenied"}
    print({"discovery": "SUCCESS", "invocation": "IAM_DENY"})
else:
    raise AssertionError("A descoberta concedeu invocação indevida")

### Invocar com a sessão autorizada

A mesma capacidade pode ser executada pela sessão completa do consumidor. A resposta deve conter o cliente sintético esperado.

In [ ]:
result = registry.invoke(record)
assert result["result"] == {"customer_id": "C001", "risk": "baixo"}
print(result)

### Evidências e encerramento

Registre IDs de requisição e decisões, sem tokens ou senhas. Os recursos são compartilhados entre aulas: a remoção pertence ao pipeline de limpeza do projeto e deve respeitar essa dependência.